In [1]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

RAW_DATA_PATH = DATA_RAW_DIR / "creditcard.csv"
CONFIG_PATH = CONFIG_DIR / "config.yaml"
CLEANING_REPORT_PATH = REPORTS_DIR / "data_cleaning_decision_log.md"
DATA_QUALITY_REPORT_PATH = REPORTS_DIR / "data_quality_report.md"

TARGET_COLUMN = "Class"
RANDOM_SEED = 42

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

EXPECTED_COLUMNS = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount", "Class"]
FEATURE_COLUMNS = [column for column in EXPECTED_COLUMNS if column != TARGET_COLUMN]

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Raw dataset path: %s", RAW_DATA_PATH)

2026-11-01 09:32:57,175 | INFO | Project root: c:\Users\use\Desktop\Credit Card Fraud Detection
2026-11-01 09:32:57,176 | INFO | Raw dataset path: c:\Users\use\Desktop\Credit Card Fraud Detection\data\raw\creditcard.csv


In [3]:
PROJECT_CONFIG = {
    "project": {
        "name": "fraud-detection-anomaly-analysis",
        "random_seed": RANDOM_SEED,
    },
    "data": {
        "raw_file": "data/raw/creditcard.csv",
        "processed_directory": "data/processed",
        "target_column": TARGET_COLUMN,
        "feature_columns": FEATURE_COLUMNS,
    },
    "split": {
        "train_size": TRAIN_SIZE,
        "validation_size": VALIDATION_SIZE,
        "test_size": TEST_SIZE,
        "stratify_by_target": True,
        "remove_exact_duplicates": True,
    },
    "preprocessing": {
        "supervised_scaled_features": FEATURE_COLUMNS,
        "anomaly_scaled_features": FEATURE_COLUMNS,
    },
}

with CONFIG_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(PROJECT_CONFIG, file, sort_keys=False)

logger.info("Project configuration saved to: %s", CONFIG_PATH)

2026-11-01 09:33:49,532 | INFO | Project configuration saved to: c:\Users\use\Desktop\Credit Card Fraud Detection\configs\config.yaml


In [4]:
def load_and_validate_raw_dataset(
    file_path: Path,
    expected_columns: list[str],
    target_column: str,
) -> pd.DataFrame:
    """Load the raw dataset and validate its schema and target labels."""
    if not file_path.exists():
        raise FileNotFoundError(f"Raw dataset not found: {file_path}")

    dataframe = pd.read_csv(file_path)

    if dataframe.empty:
        raise ValueError("The raw dataset is empty.")

    missing_columns = set(expected_columns) - set(dataframe.columns)
    unexpected_columns = set(dataframe.columns) - set(expected_columns)

    if missing_columns:
        raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

    if unexpected_columns:
        raise ValueError(f"Unexpected columns found: {sorted(unexpected_columns)}")

    if target_column not in dataframe.columns:
        raise ValueError(f"Target column '{target_column}' is missing.")

    invalid_target_values = set(dataframe[target_column].unique()) - {0, 1}
    if invalid_target_values:
        raise ValueError(
            f"Target column contains invalid values: {sorted(invalid_target_values)}"
        )

    numerical_columns = dataframe.select_dtypes(include=np.number).columns
    if len(numerical_columns) != dataframe.shape[1]:
        raise ValueError("All dataset columns must be numeric.")

    if dataframe.isnull().any().any():
        raise ValueError(
            "Missing values were detected. Cleaning policy must be reviewed before processing."
        )

    if np.isinf(dataframe.select_dtypes(include=np.number)).any().any():
        raise ValueError("Non-finite values were detected in the dataset.")

    return dataframe


raw_dataframe = load_and_validate_raw_dataset(
    file_path=RAW_DATA_PATH,
    expected_columns=EXPECTED_COLUMNS,
    target_column=TARGET_COLUMN,
)

logger.info(
    "Raw dataset validated: %s rows, %s columns",
    raw_dataframe.shape[0],
    raw_dataframe.shape[1],
)

2026-11-01 09:34:11,409 | INFO | Raw dataset validated: 284807 rows, 31 columns


In [5]:
def analyze_duplicates(
    dataframe: pd.DataFrame,
    target_column: str,
) -> dict:
    """Analyze fully duplicated rows and their distribution across classes."""
    duplicate_mask = dataframe.duplicated(keep=False)
    duplicate_rows = dataframe.loc[duplicate_mask]

    duplicate_class_counts = (
        duplicate_rows[target_column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    return {
        "total_rows": int(len(dataframe)),
        "fully_duplicated_row_count": int(len(duplicate_rows)),
        "duplicate_row_percentage": float(len(duplicate_rows) / len(dataframe) * 100),
        "duplicate_rows_by_class": {
            str(key): int(value) for key, value in duplicate_class_counts.items()
        },
        "unique_duplicate_groups": int(
            duplicate_rows.drop_duplicates().shape[0]
        ),
    }


duplicate_analysis = analyze_duplicates(
    dataframe=raw_dataframe,
    target_column=TARGET_COLUMN,
)

print("Duplicate-row analysis:")
print(json.dumps(duplicate_analysis, indent=2))

Duplicate-row analysis:
{
  "total_rows": 284807,
  "fully_duplicated_row_count": 1854,
  "duplicate_row_percentage": 0.6509671461726713,
  "duplicate_rows_by_class": {
    "0": 1822,
    "1": 32
  },
  "unique_duplicate_groups": 773
}


In [6]:
def remove_exact_duplicates(
    dataframe: pd.DataFrame,
    target_column: str,
) -> tuple[pd.DataFrame, dict]:
    """Remove exact duplicate rows while preserving the first occurrence."""
    initial_row_count = len(dataframe)
    initial_class_counts = dataframe[target_column].value_counts().sort_index().to_dict()

    cleaned_dataframe = dataframe.drop_duplicates(keep="first").copy()

    final_row_count = len(cleaned_dataframe)
    final_class_counts = cleaned_dataframe[target_column].value_counts().sort_index().to_dict()

    removal_summary = {
        "decision": "Remove exact duplicate rows, retaining the first occurrence.",
        "justification": (
            "Rows are identical across every feature and the target label. "
            "Retaining them could allow identical transactions to appear in both "
            "training and evaluation partitions, weakening split independence."
        ),
        "rows_before": initial_row_count,
        "rows_after": final_row_count,
        "rows_removed": initial_row_count - final_row_count,
        "class_counts_before": {
            str(key): int(value) for key, value in initial_class_counts.items()
        },
        "class_counts_after": {
            str(key): int(value) for key, value in final_class_counts.items()
        },
    }

    return cleaned_dataframe, removal_summary


cleaned_dataframe, duplicate_removal_summary = remove_exact_duplicates(
    dataframe=raw_dataframe,
    target_column=TARGET_COLUMN,
)

print("Duplicate removal summary:")
print(json.dumps(duplicate_removal_summary, indent=2))

Duplicate removal summary:
{
  "decision": "Remove exact duplicate rows, retaining the first occurrence.",
  "justification": "Rows are identical across every feature and the target label. Retaining them could allow identical transactions to appear in both training and evaluation partitions, weakening split independence.",
  "rows_before": 284807,
  "rows_after": 283726,
  "rows_removed": 1081,
  "class_counts_before": {
    "0": 284315,
    "1": 492
  },
  "class_counts_after": {
    "0": 283253,
    "1": 473
  }
}


In [7]:
def validate_cleaned_dataset(
    dataframe: pd.DataFrame,
    expected_columns: list[str],
    target_column: str,
) -> None:
    """Validate the cleaned dataset before splitting."""
    if dataframe.empty:
        raise ValueError("Cleaned dataset is empty.")

    if list(dataframe.columns) != expected_columns:
        raise ValueError(
            "Cleaned dataset columns do not match the expected schema or ordering."
        )

    if dataframe.duplicated().any():
        raise ValueError("Exact duplicate rows remain after cleaning.")

    if dataframe[target_column].isnull().any():
        raise ValueError("Target column contains missing values.")

    if set(dataframe[target_column].unique()) - {0, 1}:
        raise ValueError("Target column contains invalid labels.")

    if dataframe.isnull().any().any():
        raise ValueError("Cleaned dataset contains missing values.")

    logger.info(
        "Cleaned dataset validated: %s rows, %s columns",
        dataframe.shape[0],
        dataframe.shape[1],
    )


validate_cleaned_dataset(
    dataframe=cleaned_dataframe,
    expected_columns=EXPECTED_COLUMNS,
    target_column=TARGET_COLUMN,
)

2026-11-01 09:35:31,180 | INFO | Cleaned dataset validated: 283726 rows, 31 columns


In [8]:
def separate_features_and_target(
    dataframe: pd.DataFrame,
    feature_columns: list[str],
    target_column: str,
) -> tuple[pd.DataFrame, pd.Series]:
    """Create a leakage-safe feature matrix and target Series."""
    missing_features = set(feature_columns) - set(dataframe.columns)
    if missing_features:
        raise ValueError(f"Missing feature columns: {sorted(missing_features)}")

    if target_column in feature_columns:
        raise ValueError("Target column must not be included in feature columns.")

    features = dataframe[feature_columns].copy()
    target = dataframe[target_column].copy()

    if features.shape[0] != target.shape[0]:
        raise ValueError("Feature matrix and target vector have different lengths.")

    return features, target


X = separate_features_and_target(
    dataframe=cleaned_dataframe,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
)[0]

y = separate_features_and_target(
    dataframe=cleaned_dataframe,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
)[1]

logger.info("Feature matrix shape: %s", X.shape)
logger.info("Target vector shape: %s", y.shape)
logger.info("Target class counts: %s", y.value_counts().sort_index().to_dict())

2026-11-01 09:35:56,673 | INFO | Feature matrix shape: (283726, 30)
2026-11-01 09:35:56,676 | INFO | Target vector shape: (283726,)
2026-11-01 09:35:56,681 | INFO | Target class counts: {0: 283253, 1: 473}


In [9]:
def create_train_validation_test_splits(
    features: pd.DataFrame,
    target: pd.Series,
    train_size: float,
    validation_size: float,
    test_size: float,
    random_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    """Create reproducible, stratified, disjoint train/validation/test splits."""
    split_total = train_size + validation_size + test_size

    if not np.isclose(split_total, 1.0):
        raise ValueError(
            f"Split proportions must sum to 1.0, but received {split_total}."
        )

    X_train, X_temp, y_train, y_temp = train_test_split(
        features,
        target,
        train_size=train_size,
        stratify=target,
        random_state=random_seed,
    )

    relative_validation_size = validation_size / (validation_size + test_size)

    X_validation, X_test, y_validation, y_test = train_test_split(
        X_temp,
        y_temp,
        train_size=relative_validation_size,
        stratify=y_temp,
        random_state=random_seed,
    )

    return X_train, X_validation, X_test, y_train, y_validation, y_test


X_train, X_validation, X_test, y_train, y_validation, y_test = (
    create_train_validation_test_splits(
        features=X,
        target=y,
        train_size=TRAIN_SIZE,
        validation_size=VALIDATION_SIZE,
        test_size=TEST_SIZE,
        random_seed=RANDOM_SEED,
    )
)

logger.info("Train shape: %s", X_train.shape)
logger.info("Validation shape: %s", X_validation.shape)
logger.info("Test shape: %s", X_test.shape)

2026-11-01 09:36:05,338 | INFO | Train shape: (198608, 30)
2026-11-01 09:36:05,339 | INFO | Validation shape: (42559, 30)
2026-11-01 09:36:05,340 | INFO | Test shape: (42559, 30)


In [10]:
def validate_splits(
    X_train: pd.DataFrame,
    X_validation: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_validation: pd.Series,
    y_test: pd.Series,
) -> dict:
    """Verify disjoint splits, row counts, and class distributions."""
    train_indices = set(X_train.index)
    validation_indices = set(X_validation.index)
    test_indices = set(X_test.index)

    overlaps = {
        "train_validation": len(train_indices & validation_indices),
        "train_test": len(train_indices & test_indices),
        "validation_test": len(validation_indices & test_indices),
    }

    if any(overlaps.values()):
        raise ValueError(f"Data leakage detected across splits: {overlaps}")

    if TARGET_COLUMN in X_train.columns:
        raise ValueError("Target column found in training features.")

    split_summary = {
        "train": {
            "row_count": int(len(X_train)),
            "class_counts": y_train.value_counts().sort_index().to_dict(),
            "fraud_percentage": float(y_train.mean() * 100),
        },
        "validation": {
            "row_count": int(len(X_validation)),
            "class_counts": y_validation.value_counts().sort_index().to_dict(),
            "fraud_percentage": float(y_validation.mean() * 100),
        },
        "test": {
            "row_count": int(len(X_test)),
            "class_counts": y_test.value_counts().sort_index().to_dict(),
            "fraud_percentage": float(y_test.mean() * 100),
        },
        "split_overlaps": overlaps,
    }

    return split_summary


split_summary = validate_splits(
    X_train=X_train,
    X_validation=X_validation,
    X_test=X_test,
    y_train=y_train,
    y_validation=y_validation,
    y_test=y_test,
)

print("Split summary:")
print(json.dumps(split_summary, indent=2))

Split summary:
{
  "train": {
    "row_count": 198608,
    "class_counts": {
      "0": 198277,
      "1": 331
    },
    "fraud_percentage": 0.16665995327479255
  },
  "validation": {
    "row_count": 42559,
    "class_counts": {
      "0": 42488,
      "1": 71
    },
    "fraud_percentage": 0.1668272280833666
  },
  "test": {
    "row_count": 42559,
    "class_counts": {
      "0": 42488,
      "1": 71
    },
    "fraud_percentage": 0.1668272280833666
  },
  "split_overlaps": {
    "train_validation": 0,
    "train_test": 0,
    "validation_test": 0
  }
}


In [11]:
def save_processed_splits(
    X_train: pd.DataFrame,
    X_validation: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_validation: pd.Series,
    y_test: pd.Series,
    output_directory: Path,
) -> dict:
    """Save feature and target splits in Parquet format."""
    split_paths = {
        "X_train": output_directory / "train" / "X_train.parquet",
        "y_train": output_directory / "train" / "y_train.parquet",
        "X_validation": output_directory / "validation" / "X_validation.parquet",
        "y_validation": output_directory / "validation" / "y_validation.parquet",
        "X_test": output_directory / "test" / "X_test.parquet",
        "y_test": output_directory / "test" / "y_test.parquet",
    }

    for path in split_paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)

    X_train.to_parquet(split_paths["X_train"], index=True)
    y_train.to_frame().to_parquet(split_paths["y_train"], index=True)

    X_validation.to_parquet(split_paths["X_validation"], index=True)
    y_validation.to_frame().to_parquet(split_paths["y_validation"], index=True)

    X_test.to_parquet(split_paths["X_test"], index=True)
    y_test.to_frame().to_parquet(split_paths["y_test"], index=True)

    return {name: str(path) for name, path in split_paths.items()}


saved_split_paths = save_processed_splits(
    X_train=X_train,
    X_validation=X_validation,
    X_test=X_test,
    y_train=y_train,
    y_validation=y_validation,
    y_test=y_test,
    output_directory=DATA_PROCESSED_DIR,
)

print("Saved processed split files:")
print(json.dumps(saved_split_paths, indent=2))

Saved processed split files:
{
  "X_train": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\train\\X_train.parquet",
  "y_train": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\train\\y_train.parquet",
  "X_validation": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\validation\\X_validation.parquet",
  "y_validation": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\validation\\y_validation.parquet",
  "X_test": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\test\\X_test.parquet",
  "y_test": "c:\\Users\\use\\Desktop\\Credit Card Fraud Detection\\data\\processed\\test\\y_test.parquet"
}


In [12]:
def build_split_metadata(
    split_summary: dict,
    duplicate_removal_summary: dict,
    random_seed: int,
) -> dict:
    """Create reproducible metadata describing the data preparation process."""
    return {
        "source_file": str(RAW_DATA_PATH),
        "cleaning": duplicate_removal_summary,
        "split": {
            "method": "stratified_random_split",
            "random_seed": random_seed,
            "train_size": TRAIN_SIZE,
            "validation_size": VALIDATION_SIZE,
            "test_size": TEST_SIZE,
            "summary": split_summary,
        },
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "preprocessing_status": (
            "Preprocessing pipelines are defined but not fitted in this phase. "
            "They must be fitted only on training data in later modelling phases."
        ),
    }


split_metadata = build_split_metadata(
    split_summary=split_summary,
    duplicate_removal_summary=duplicate_removal_summary,
    random_seed=RANDOM_SEED,
)

split_metadata_path = DATA_PROCESSED_DIR / "split_metadata.json"

with split_metadata_path.open("w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=2, default=str)

logger.info("Split metadata saved to: %s", split_metadata_path)

2026-11-01 09:39:20,292 | INFO | Split metadata saved to: c:\Users\use\Desktop\Credit Card Fraud Detection\data\processed\split_metadata.json


In [13]:
def create_scaled_numerical_pipeline() -> Pipeline:
    """Create a numerical preprocessing pipeline for scale-sensitive models."""
    return Pipeline(
        steps=[
            ("standard_scaler", StandardScaler()),
        ]
    )


def create_unscaled_numerical_pipeline() -> Pipeline:
    """Create a numerical preprocessing pipeline for tree-based models."""
    return Pipeline(
        steps=[
            ("identity_preprocessing", "passthrough"),
        ]
    )


supervised_scaled_pipeline = create_scaled_numerical_pipeline()
tree_based_pipeline = create_unscaled_numerical_pipeline()
anomaly_scaled_pipeline = create_scaled_numerical_pipeline()

preprocessing_definitions = {
    "logistic_regression": "StandardScaler fitted on training data only.",
    "random_forest": "No scaling required.",
    "lightgbm": "No scaling required.",
    "isolation_forest": "Scaling may be investigated; baseline uses scaled features.",
    "local_outlier_factor": "StandardScaler required because LOF is distance-based.",
}

print("Preprocessing definitions:")
print(json.dumps(preprocessing_definitions, indent=2))

Preprocessing definitions:
{
  "logistic_regression": "StandardScaler fitted on training data only.",
  "random_forest": "No scaling required.",
  "lightgbm": "No scaling required.",
  "isolation_forest": "Scaling may be investigated; baseline uses scaled features.",
  "local_outlier_factor": "StandardScaler required because LOF is distance-based."
}


In [14]:
def write_data_quality_report(
    raw_dataframe: pd.DataFrame,
    cleaned_dataframe: pd.DataFrame,
    duplicate_analysis: dict,
    split_summary: dict,
    output_path: Path,
) -> None:
    """Write a concise data-quality report based on actual execution results."""
    report = f"""# Data Quality Report

## Source Dataset

- Source file: `{RAW_DATA_PATH.name}`
- Raw rows: {len(raw_dataframe):,}
- Raw columns: {raw_dataframe.shape[1]}
- Missing values: {int(raw_dataframe.isnull().sum().sum())}
- Exact duplicate rows involved in duplication groups: {duplicate_analysis['fully_duplicated_row_count']:,}

## Cleaning Decision

{duplicate_removal_summary['decision']}

**Justification:** {duplicate_removal_summary['justification']}

- Rows before cleaning: {duplicate_removal_summary['rows_before']:,}
- Rows after cleaning: {duplicate_removal_summary['rows_after']:,}
- Rows removed: {duplicate_removal_summary['rows_removed']:,}
- Legitimate transactions before cleaning: {duplicate_removal_summary['class_counts_before']['0']:,}
- Legitimate transactions after cleaning: {duplicate_removal_summary['class_counts_after']['0']:,}
- Fraudulent transactions before cleaning: {duplicate_removal_summary['class_counts_before']['1']:,}
- Fraudulent transactions after cleaning: {duplicate_removal_summary['class_counts_after']['1']:,}

## Split Integrity

- Training rows: {split_summary['train']['row_count']:,}
- Validation rows: {split_summary['validation']['row_count']:,}
- Test rows: {split_summary['test']['row_count']:,}
- Train-validation overlap: {split_summary['split_overlaps']['train_validation']}
- Train-test overlap: {split_summary['split_overlaps']['train_test']}
- Validation-test overlap: {split_summary['split_overlaps']['validation_test']}

## Preprocessing Policy

- The target column `{TARGET_COLUMN}` is excluded from all feature matrices.
- Scaling is not applied in this notebook.
- Any scaler or imputer must be fitted using training data only.
- Validation and test data must be transformed using already-fitted training components.
- No outliers are removed in this phase.

## Limitations

- `V1` to `V28` are anonymized PCA-transformed features and must not be assigned business meanings.
- `Time` is elapsed seconds from the first transaction, not a calendar timestamp.
- Extreme values are retained because they may be legitimate transactions or useful signals.
"""
    output_path.write_text(report, encoding="utf-8")


write_data_quality_report(
    raw_dataframe=raw_dataframe,
    cleaned_dataframe=cleaned_dataframe,
    duplicate_analysis=duplicate_analysis,
    split_summary=split_summary,
    output_path=DATA_QUALITY_REPORT_PATH,
)

CLEANING_REPORT_PATH.write_text(
    "# Data Cleaning Decision Log\n\n"
    "## Exact Duplicate Rows\n\n"
    f"- Decision: {duplicate_removal_summary['decision']}\n"
    f"- Justification: {duplicate_removal_summary['justification']}\n"
    f"- Rows removed: {duplicate_removal_summary['rows_removed']:,}\n"
    f"- Rows remaining: {duplicate_removal_summary['rows_after']:,}\n\n"
    "## Outliers\n\n"
    "- No outliers were removed. Extreme values may represent valid transactions or useful fraud-related signals.\n\n"
    "## Missing Values\n\n"
    "- No missing values were found; therefore, no imputation was applied.\n\n"
    "## Data Types\n\n"
    "- All columns were already numeric; no type conversion was necessary.\n",
    encoding="utf-8",
)

logger.info("Data quality report saved to: %s", DATA_QUALITY_REPORT_PATH)
logger.info("Cleaning decision log saved to: %s", CLEANING_REPORT_PATH)

2026-11-01 09:39:31,293 | INFO | Data quality report saved to: c:\Users\use\Desktop\Credit Card Fraud Detection\reports\data_quality_report.md
2026-11-01 09:39:31,295 | INFO | Cleaning decision log saved to: c:\Users\use\Desktop\Credit Card Fraud Detection\reports\data_cleaning_decision_log.md


In [15]:
required_artifacts = [
    *saved_split_paths.values(),
    split_metadata_path,
    CONFIG_PATH,
    DATA_QUALITY_REPORT_PATH,
    CLEANING_REPORT_PATH,
]

missing_artifacts = [path for path in required_artifacts if not Path(path).exists()]

if missing_artifacts:
    raise FileNotFoundError(
        f"Missing Phase 3 artifacts: {[str(path) for path in missing_artifacts]}"
    )

logger.info("All required Phase 3 artifacts were generated successfully.")
print("Phase 3 data cleaning and preparation completed successfully.")

2026-11-01 09:39:40,669 | INFO | All required Phase 3 artifacts were generated successfully.


Phase 3 data cleaning and preparation completed successfully.
